In [ ]:
!pip install huggingface_hub
!huggingface-cli login
#!huggingface-cli repo create ner_model_bert_base



    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To login, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: write).
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in 

In [ ]:
!pip install datasets
!pip install transformers[torch]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4

In [ ]:
import pandas as pd
import numpy as np
import torch
import random
import ast
from datasets import Dataset, DatasetDict, Features, Sequence, ClassLabel, Value
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Establecer semilla aleatoria para la reproducibilidad

# Ruta al archivo CSV
csv_path = "/content/drive/MyDrive/Colab Notebooks/etiquetado.csv"

# Cargar DataFrame desde el archivo CSV y convertirlo a dict
pd_df = pd.read_csv(csv_path)

# Convertir las columnas 'tokens' y 'ner_tags' de cadenas a listas
pd_df['tokens'] = pd_df['tokens'].apply(ast.literal_eval)
pd_df['ner_tags'] = pd_df['ner_tags'].apply(ast.literal_eval)

dict_corpus = pd_df.to_dict(orient='list')

# Solución 1: Agregar 'TRA_ID' a features
features = Features({
    'tokens': Sequence(Value('string')),
    'ner_tags': Sequence(ClassLabel(names=["O", "B-PER", "I-PER", "B-LOC", "I-LOC"])),
})

del dict_corpus['TRA_ID']

# Convertir el corpus a Dataset de la biblioteca `datasets`
dataset = Dataset.from_dict(dict_corpus, features=features)

# Dividir el dataset en entrenamiento (80%), validación (10%) y testeo (10%)
train_valid_test_split = dataset.train_test_split(test_size=0.2)
valid_test_split = train_valid_test_split["test"].train_test_split(test_size=0.5)
dataset = DatasetDict({
    'train': train_valid_test_split['train'],
    'validation': valid_test_split['train'],
    'test': valid_test_split['test']
})

# Cargar el modelo y el tokenizador
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
model = AutoModelForTokenClassification.from_pretrained("dccuchile/bert-base-spanish-wwm-cased",
    num_labels=len(dataset['train'].features['ner_tags'].feature.names),
    ignore_mismatched_sizes=True
).to(device)

# Obtener la longitud máxima permitida por el modelo
max_length = tokenizer.model_max_length

# Función para tokenizar y alinear etiquetas
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding='max_length',  # Asegura el padding al largo máximo permitido por el modelo
        max_length=max_length,
        is_split_into_words=True
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        # Ajustar etiquetas a max_length
        if len(label_ids) < max_length:
            label_ids.extend([-100] * (max_length - len(label_ids)))
        else:
            label_ids = label_ids[:max_length]
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenizar el dataset
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

# Crear el Data Collator
data_collator = DataCollatorForTokenClassification(
    tokenizer,
    padding='max_length',
    max_length=max_length
)

# Parámetros de entrenamiento
training_args = TrainingArguments(
    output_dir="./results1",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    save_strategy="epoch",
    load_best_model_at_end=True
)

# Métricas de evaluación
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Flatten the lists
    true_labels = [dataset['train'].features['ner_tags'].feature.names[l] for label in labels for l in label if l != -100]
    true_predictions = [dataset['train'].features['ner_tags'].feature.names[p] for prediction, label in zip(predictions, labels) for p, l in zip(prediction, label) if l != -100]

    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, true_predictions, labels=dataset['train'].features['ner_tags'].feature.names, average='macro', zero_division=0)
    accuracy = accuracy_score(true_labels, true_predictions)

    return {
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# Configuración del entrenador
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Entrenar el modelo
trainer.train()

# Evaluar el modelo en el conjunto de validación
validation_results = trainer.evaluate()
print("Resultados de Validación:", validation_results)

# Evaluar el modelo en el conjunto de prueba
test_results = trainer.evaluate(tokenized_dataset["test"])
print("Resultados de Prueba:", test_results)

# Guardar el modelo
model.save_pretrained("./ner_model_bert_base")
tokenizer.save_pretrained("./ner_model_bert_base")

model.push_to_hub("ner_model_bert_base")
tokenizer.push_to_hub("ner_model_bert_base")



# Función para predecir entidades distintas de 'O'
def predict_non_O_entities(dataset, tokenizer, model, label_list, device):
    results = []
    softmax = torch.nn.Softmax(dim=-1)
    for example in dataset:
        tokens = example["tokens"]
        inputs = tokenizer(tokens, padding='max_length', truncation=True, return_tensors="pt", is_split_into_words=True)
        inputs = {key: value.to(device) for key, value in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits
        logits = logits[0].cpu().numpy()

        probabilities = softmax(torch.from_numpy(logits))
        scores = np.max(probabilities.numpy(), axis=-1)

        predictions = np.argmax(logits, axis=-1)
        predicted_labels = [label_list[p] for p in predictions[:len(tokens)]]
        predicted_scores = scores[:len(tokens)]

        result = []
        for token, label, score in zip(tokens, predicted_labels, predicted_scores):
            if label != "O":
                result.append({
                    "token": token,
                    "label": label,
                    "score": score
                })
        if result:
            results.append(result)
    return results

# Obtener predicciones con puntajes para entidades distintas de 'O' en el conjunto de prueba
predictions_non_O = predict_non_O_entities(tokenized_dataset["test"], tokenizer, model, dataset['train'].features['ner_tags'].feature.names, device)

# Mostrar predicciones detalladas para entidades distintas de 'O'
for example in predictions_non_O[:3]:  # Mostrar los primeros 3 ejemplos
    for token_info in example:
        print(f"Entidad: {token_info['token']}, Tipo: {token_info['label']}, Score: {token_info['score']:.4f}")
    print()


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

...

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.190844,0.933808,0.729802,0.772082,0.710784
2,No log,0.145764,0.948036,0.799777,0.791384,0.811766
3,No log,0.147495,0.949892,0.816932,0.789008,0.849702


Resultados de Validación: {'eval_loss': 0.14576351642608643, 'eval_accuracy': 0.9480358799876276, 'eval_f1': 0.7997773496459772, 'eval_precision': 0.7913839599571777, 'eval_recall': 0.8117659013006897, 'eval_runtime': 85.1331, 'eval_samples_per_second': 0.599, 'eval_steps_per_second': 0.082, 'epoch': 3.0}
Resultados de Prueba: {'eval_loss': 0.0834042951464653, 'eval_accuracy': 0.9739766081871345, 'eval_f1': 0.8898766824816503, 'eval_precision': 0.8801934151701145, 'eval_recall': 0.9001920589792443, 'eval_runtime': 85.3863, 'eval_samples_per_second': 0.597, 'eval_steps_per_second': 0.082, 'epoch': 3.0}


HfHubHTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/api/repos/create (Request ID: Root=1-666f92fd-33c41fe2054f55746e777820;bf887048-da3e-4921-a0be-0794f9061a85)

Invalid username or password.

In [ ]:
# prompt: : Using the `Trainer` with `PyTorch` requires `accelerate>=0.21.0`: Please run `pip install transformers[torch]` or `pip install accelerate -U`

!pip install datasets
!pip install transformers[torch]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 14.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 21.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 2